In [2]:
import kymnasium as kym
import gymnasium as gym


env = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='none'
)
env.reset()

({'mario': array([288., 768., 336., 816.,   0.], dtype=float32),
  'blurps': array([[0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.

# Constants

In [3]:
REPLAY_BUFFER_SIZE = 100000
EPSILON_MIN = 0.2
EPSILON_DECAY = 0.9995
INIT_EXPLORATION = 50000
RANDOM_SEED = 42
INTERVAL_UPDATE = 8
BATCH_SIZE = 32
TAU = 0.005
GAMMA = 0.99995
ALPHA = 0.6
BETA = 1.0
LEARNING_RATE = 0.0003
STATE_DIM = 3 + 30 * 5
STATE_SEQ = 8
ACTION_DIM = 3
MAX_EPISODES = 10000
N_MONITOR = 100
WIDTH, HEIGHT = 624, 912

In [4]:
class Tracker:
    def __init__(self, monitor: int):
        self._monitor = monitor
        self._values = []

    def update(self, value):
        if len(self._values) > self._monitor:
            del self._values[:1]

        self._values.append(value)

    @property
    def avg_(self):
        return np.mean(self._values)

In [5]:
from tensorflow import keras
import numpy as np


class ReplayBuffer:
    def __init__(self, capacity):
        self._capacity = capacity
        self._states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM))
        self._actions = np.zeros(shape=(self._capacity, ACTION_DIM))
        self._rewards = np.zeros(shape=(self._capacity,))
        self._next_states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM))
        self._dones = np.zeros(shape=(self._capacity,))
        self._priority = np.ones(shape=(self._capacity,))
        self._size = 0
        self._index = 0
        self._random = np.random.default_rng(RANDOM_SEED)

    @property
    def size_(self):
        return self._size

    def push(self, state, action, reward, next_state, done, priority):
        self._states[self._index] = state
        self._actions[self._index] = action
        self._rewards[self._index] = reward
        self._next_states[self._index] = next_state
        self._dones[self._index] = done
        self._priority[self._index] = priority
        self._index = (self._index + 1) % self._capacity
        self._size = min(self._size + 1, self._capacity)

    def sample(self, batch_size):
        priority = np.power(self._priority[:self._size], ALPHA)
        prob = priority / np.sum(priority)
        indices = self._random.choice(self._size, size=batch_size, p=prob)

        return (
            indices,
            self._states[indices],
            self._actions[indices],
            self._rewards[indices],
            self._next_states[indices],
            self._dones[indices],
            prob
        )

    def update_priority(self, indices, priority):
        self._priority[indices] = priority

In [6]:
from tensorflow import keras


def build_network():
    inputs = keras.Input(
        shape=(8, 3 + 30 * 5)
    )
    x = keras.layers.Conv1D(
        filters=32,
        kernel_size=4,
        activation='relu',
        strides=2,
        kernel_initializer='he_normal',
    )(inputs)

    x = keras.layers.Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)

    x = keras.layers.Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(
        units=512,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)
    value = keras.layers.Dense(
        units=1,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(x)
    advantage = keras.layers.Dense(
        units=3,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(x)

    return keras.models.Model(inputs=inputs, outputs=[value, advantage])


behavior_network = build_network()
target_network = build_network()

In [17]:
import tensorflow as tf
from tensorflow import keras


objective = keras.losses.Huber(reduction=None)
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)


def train(states, actions, rewards, next_states, dones, prob):
    next_state_value, next_advantage = target_network(next_states)
    next_mean_advantage = keras.ops.mean(next_advantage, axis=1, keepdims=True)
    next_Q = keras.ops.max(
        next_advantage - next_mean_advantage + next_state_value,
        axis=1,
        keepdims=True
    )

    targets = rewards + (1 - dones) * GAMMA * next_Q

    weights = keras.ops.power(REPLAY_BUFFER_SIZE * prob, -BETA)
    weights /= keras.ops.max(weights)
    weights = keras.ops.cast(weights, 'float32')

    with tf.GradientTape() as tape:
        state_value, advantage = behavior_network(states)
        mean_advantage = keras.ops.mean(advantage, axis=1, keepdims=True)
        Q = keras.ops.sum(
            (advantage - mean_advantage + state_value) * actions,
            axis=1,
            keepdims=True
        )
        priority = objective(targets, Q)
        print(state_value, Q, priority, weights)
        loss = keras.ops.mean(priority * weights)

    gradient = tape.gradient(loss, behavior_network.trainable_variables)
    optimizer.apply_gradients(zip(gradient, behavior_network.trainable_variables))

    target_weights, behavior_weights = target_network.get_weights(), behavior_network.get_weights()
    for i in range(len(target_weights)):
        target_weights[i] = TAU * behavior_weights[i] + target_weights[i] * (1.0 - TAU)
    target_network.set_weights(target_weights)

    return loss, priority

In [8]:
from tensorflow import keras


def choose_action(state):
    _, adv = behavior_network(state)
    adv = keras.ops.ravel(adv)
    return keras.ops.argmax(adv)

In [9]:
def preprocess(obs):
    mario, blurps = obs['mario'], obs['blurps']

    mario = np.array([mario[0] / WIDTH, mario[1] / HEIGHT, mario[-1] / WIDTH])
    blurps = np.array([
        [blurp[0] / WIDTH, blurp[1] / HEIGHT, blurp[4] / WIDTH, blurp[5] / HEIGHT, blurp[6] / HEIGHT] for blurp in blurps
    ])
    blurps = np.ravel(blurps)
    return np.concatenate([mario, blurps])

n

In [18]:
from tqdm.auto import tqdm


pbar = tqdm(range(MAX_EPISODES), desc='episode')
reward_tracker = Tracker(N_MONITOR)
loss_tracker = Tracker(N_MONITOR)
epsilon = 1.0

random = np.random.default_rng(RANDOM_SEED)
replay_buffer = ReplayBuffer(REPLAY_BUFFER_SIZE)
max_priority = 1.0

for episode in pbar:
    steps, total_reward = 0.0, 0.0

    done = False
    obs, _ = env.reset()
    obs = preprocess(obs)

    running_states = [
        np.zeros((STATE_DIM, )) for _ in range(STATE_SEQ)
    ]

    del running_states[:1]
    running_states.append(obs)
    state = keras.ops.expand_dims(running_states, axis=0)

    while not done:
        if episode > INIT_EXPLORATION:
            epsilon = max(epsilon * EPSILON_DECAY, EPSILON_MIN)

        if random.random() < epsilon:
            action = choose_action(state).numpy()
        else:
            action = random.choice(ACTION_DIM)

        next_obs, _, terminated, truncated, _ = env.step(action)
        next_obs = preprocess(next_obs)

        del running_states[:1]
        running_states.append(next_obs)
        next_state = keras.ops.expand_dims(running_states, axis=0)

        done = terminated or truncated
        steps += 1
        reward = 0.01 if not done else 0.0
        total_reward += reward

        replay_buffer.push(
            state, keras.ops.one_hot(action, ACTION_DIM), reward, next_state, done, max_priority
        )

        if steps % INTERVAL_UPDATE == 0 and replay_buffer.size_ >= BATCH_SIZE:
            indices, states, actions, rewards, next_states, dones, probs = replay_buffer.sample(BATCH_SIZE)
            loss, priorities = train(states, actions, rewards, next_states, dones, probs)
            replay_buffer.update_priority(indices, priorities)
            max_priority = max(max_priority, np.max(priorities))
            pbar.set_postfix({
              'loss': f'{loss:.5f}'
            })

        state = next_state

    reward_tracker.update(total_reward)
    pbar.set_postfix({
         'reward': f'{reward_tracker.avg_:.5f}'
    })


episode:   0%|          | 0/10000 [00:00<?, ?it/s, loss=0.00842]

tf.Tensor(
[[-0.1340555 ]
 [-0.061423  ]
 [-0.10380498]
 [-0.06432351]
 [-0.06179274]
 [-0.1790929 ]
 [-0.1340555 ]
 [-0.14444858]
 [-0.07572054]
 [-0.061423  ]
 [-0.04751907]
 [-0.17843476]
 [-0.06432351]
 [-0.16999742]
 [-0.061423  ]
 [-0.08174608]
 [-0.06281345]
 [-0.05383069]
 [-0.16999742]
 [-0.06432351]
 [-0.1340555 ]
 [-0.04751907]
 [-0.1790929 ]
 [-0.16283166]
 [-0.1340555 ]
 [-0.00345409]
 [-0.061423  ]
 [-0.04312244]
 [-0.07572054]
 [-0.06432351]
 [-0.10536111]
 [-0.18147475]], shape=(32, 1), dtype=float32) tf.Tensor(
[[-0.06886511]
 [-0.00881181]
 [-0.01624008]
 [-0.01693519]
 [-0.02483645]
 [-0.07295541]
 [-0.06886511]
 [-0.07827123]
 [-0.04936268]
 [-0.00881181]
 [ 0.00569262]
 [-0.09871851]
 [-0.01693519]
 [-0.10880519]
 [-0.00881181]
 [-0.02150109]
 [-0.014427  ]
 [-0.0391807 ]
 [-0.10880519]
 [-0.01693519]
 [-0.06886511]
 [ 0.00569262]
 [-0.07295541]
 [-0.07023284]
 [-0.06886511]
 [ 0.05716506]
 [-0.00881181]
 [-0.03100113]
 [-0.04936268]
 [-0.01693519]
 [-0.0401815 ]
 

InvalidArgumentError: {{function_node __wrapped__Mul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [32] vs. [40] [Op:Mul] name: 